In [1]:
import requests
import json
import pandas as pd
import numpy as np
import os
from datetime import datetime

API_KEY = None
with open("keys.json") as f:
    data = json.loads(f.read())
    API_KEY = data["STEAM_DATA_API"] 

BASE_URL = "https://steamdataapi.com/api/v1"

In [19]:
def call(method: str, call_path: str, extra_params: str | None = None, payload: dict | None = None, base_url = BASE_URL):
    """
    calls the API, stores the response to /data, returns (location of saved file, errorcode)
    """
    method = method.upper()
    body = ""
    headers = {}

    request_url = base_url + call_path + "?key=" + API_KEY 

    output_path = f"data/{call_path.replace('/', '_')}"
    output_path = output_path[:output_path.find("markets")].rstrip()

    if extra_params != None:
        request_url = request_url + "&" + extra_params
        output_path = output_path + "_" + extra_params + ".json"

    if payload:
        body = json.dumps(payload)


    if method != "GET" and payload:
        headers["Content-Type"] = "application/json"

    try:
        r = requests.request(
            method,
            request_url,
            headers=headers,
            data=body.encode() if body else None,
            timeout=15,
        )
        r.raise_for_status()
        data_json = r.json()

        if len(data_json) == 0:
            return output_path, "RESPONSE EMPTY"

        file_opening_mode = "x"

        if (os.path.exists(output_path)):
            file_opening_mode = "w"
        
        with open(output_path, file_opening_mode) as f:
            json.dump(data_json, f, indent=4)

        return output_path, None
    except requests.RequestException as e:
        detail = e.response.text if e.response is not None else "no body"
        return output_path, f"{e} — {detail}"


def getJsonFileasFlatMappedDataFrame(file_path: str):
    return pd.json_normalize(json.loads(open(file_path).read()))

def getJsonObjAsFlatMappedDataFrame(object):
    return pd.json_normalize(object)

def getData(call_name: str, file_name: str):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
    }

    try:
        response = requests.get(
            call_name,
            headers=headers,
        )

        response.raise_for_status()
        data = response.json()
        file_name = file_name + "_" + str(datetime.now()) + ".json"
        with open(file_name, "x") as f:
            json.dump(data,f, indent=4)

        print(f"File: {file_name}" +  ", Size: " + str(os.path.getsize(file_name)/1000) + "KB" )
        return 0
    
    except requests.RequestException as e:
        detail = e.response.text if e.response is not None else "no body"
        print(f"{e} — {detail}")
        return 1

In [ ]:
call_path = "steam/api/items"

for game_name in ["cs2", "dota", "tf2"]:
    game_param = f"game={game_name}"
    for pagenum in range(0,5):
        full_params = game_param + '&pretty=1&' + 'page=' + str(pagenum)
        location, errorcode = call("GET", call_path, full_params)
        if (errorcode != None):
            print(errorcode)
            break
        print(f"Nr: {pagenum}, File: {location}" +  ", Size: " + str(os.path.getsize(location)/1000) + "KB" )

In [20]:
call_path = "https://steamdataapi.com/api/v1/items/all?game=cs2&markets=1&variants=1&currency=USD"
file_name = "data/all_items_no_info"

errorcode = getData(call_path, file_name)

File: data/all_items_no_info_2026-11-17 18:05:45.603960.json, Size: 132701.271KB


In [39]:
for pageNum in range(32,62):
    call_path = f"https://steamdataapi.com/api/v1/buy-orders?game=cs2&limit=500&page={pageNum}&currency=USD"
    file_name = f"data/buy_orders_page_{pageNum}"
    errorcode = getData(call_path, file_name)
    if errorcode == 1:
        break

File: data/buy_orders_page_32_2026-11-17 18:25:11.552795.json, Size: 1242.177KB
File: data/buy_orders_page_33_2026-11-17 18:25:11.950522.json, Size: 1231.645KB
File: data/buy_orders_page_34_2026-11-17 18:25:12.494041.json, Size: 1262.811KB
File: data/buy_orders_page_35_2026-11-17 18:25:12.917447.json, Size: 1232.792KB
File: data/buy_orders_page_36_2026-11-17 18:25:13.710480.json, Size: 1218.238KB
File: data/buy_orders_page_37_2026-11-17 18:25:14.100284.json, Size: 939.892KB
File: data/buy_orders_page_38_2026-11-17 18:25:14.497129.json, Size: 1214.498KB
File: data/buy_orders_page_39_2026-11-17 18:25:15.064095.json, Size: 1265.166KB
File: data/buy_orders_page_40_2026-11-17 18:25:15.632678.json, Size: 1246.619KB
File: data/buy_orders_page_41_2026-11-17 18:25:16.262469.json, Size: 1261.832KB
File: data/buy_orders_page_42_2026-11-17 18:25:16.688093.json, Size: 1239.975KB
File: data/buy_orders_page_43_2026-11-17 18:25:17.103591.json, Size: 1272.777KB
File: data/buy_orders_page_44_2026-11-17 

# Datenexploration der Verkaufspreise

In [ ]:

def verkaufs_angebote_normalizen(df: pd.DataFrame):
    """
    1. schauen ob varianten existieren oder nicht
    2. wenn varianten existieren, exploden & normalisieren, jede variante hat wieder eigene markets liste
    3. varianten namen umbennen, da priceUpdatedAt / updatedAt/capturedAt nicht gleich genannt werden 
    4. alle zusammenfügen & market exploden & normalisieren
    """


    has_variants = df["variants"].notna() & (df["variants"].str.len() > 0)
    df_no_var = df[~has_variants].drop(columns=["variants"]).copy()
    df_var = df[has_variants].drop(columns=["markets", "priceUpdatedAt"], errors="ignore").copy()  

    df_var = df_var.explode("variants").reset_index(drop=True)
    variant_details = pd.json_normalize(df_var["variants"]).drop(columns=["best","bestMarket"])
    variant_details = variant_details.rename(columns={"updatedAt": "priceUpdatedAt"})
    df_var = pd.concat([df_var.drop(columns=["variants"]), variant_details], axis=1)
    combined_df = pd.concat([df_no_var, df_var], ignore_index=True)

    combined_df = combined_df.explode("markets").dropna(subset=["markets"]).reset_index(drop=True)
    market_details = pd.json_normalize(combined_df["markets"])

    if "updatedAt" in market_details.columns:
        market_details["capturedAt"] = market_details["capturedAt"].fillna(market_details["updatedAt"])
        market_details = market_details.drop(columns=["updatedAt"])

    return pd.concat(
        [
            combined_df.drop(columns=["markets"]),
            market_details.drop(columns=["icon"], errors="ignore"),
        ],
        axis=1,
    )

In [ ]:
all_items_file_alt = json.loads(open("data/all_items_no_info_2026-11-17 13:44:47.776898.json").read())
verkaufs_angebote_alt = getJsonObjAsFlatMappedDataFrame(all_items_file_alt["data"]).drop(columns=["game","prices.best","prices.bestSource","prices.latest","prices.median","prices.buyorder","prices.real","prices.realMedian","prices.realMarket"])

all_items_file_neu = json.loads(open("data/all_items_no_info_2026-11-17 18:05:45.603960.json").read())
verkaufs_angebote_neu = getJsonObjAsFlatMappedDataFrame(all_items_file_neu["data"]).drop(columns=["game","prices.best","prices.bestSource","prices.latest","prices.median","prices.buyorder","prices.real","prices.realMedian","prices.realMarket"])


In [131]:
alt_normalized = verkaufs_angebote_normalizen(verkaufs_angebote_alt)


,marketHashName,game,sold24h,priceUpdatedAt,variant,market,price,quantity,capturedAt
0,'Blueberries' Buckshot | NSWC SEAL,cs2,17.0,2026-11-17T13:37:55.395Z,NaN,avan,936,216.0,2026-11-17T13:43:09.578Z
1,'Blueberries' Buckshot | NSWC SEAL,cs2,17.0,2026-11-17T13:37:55.395Z,NaN,buff163,897,187.0,2026-11-17T13:44:25.468Z
2,'Blueberries' Buckshot | NSWC SEAL,cs2,17.0,2026-11-17T13:37:55.395Z,NaN,c5game,926,263.0,2026-11-17T13:44:25.468Z
3,'Blueberries' Buckshot | NSWC SEAL,cs2,17.0,2026-11-17T13:37:55.395Z,NaN,csfloat,930,190.0,2026-11-17T13:40:20.942Z
4,'Blueberries' Buckshot | NSWC SEAL,cs2,17.0,2026-11-17T13:37:55.395Z,NaN,csmoney,909,95.0,2026-11-17T13:44:01.373Z
...,...,...,...,...,...,...,...,...,...
392993,★ Ursus Knife | Doppler (Minimal Wear),cs2,1.0,2026-11-17T13:44:25.468Z,Sapphire,whitemarket,72822,12.0,2026-11-17T13:30:49.598Z
392994,★ Ursus Knife | Doppler (Minimal Wear),cs2,1.0,2026-11-17T13:44:25.468Z,Sapphire,csmoney,74060,5.0,2026-11-17T13:44:01.373Z
392995,★ Ursus Knife | Doppler (Minimal Wear),cs2,1.0,2026-11-17T13:44:25.468Z,Sapphire,avan,75871,8.0,2026-11-17T13:43:09.578Z
392996,★ Ursus Knife | Doppler (Minimal Wear),cs2,1.0,2026-11-17T13:44:25.468Z,Sapphire,marketcsgo,75999,1.0,2026-11-17T13:42:00.976Z


In [97]:
markets_exploded = verkaufs_angebote_alt.explode("markets")
market_details = pd.json_normalize(markets_exploded["markets"])

market_cleaned_df = pd.concat([
    markets_exploded.drop(columns=["markets"]).reset_index(drop=True),
    market_details.drop(columns=["icon"]),
], axis=1)


In [103]:
df_has_variants = market_cleaned_df[market_cleaned_df["variants"].notnull()]

In [104]:
variants_exploded = df_has_variants.explode("variants")
variant_details = pd.json_normalize(variants_exploded["variants"])

variants_cleaned_df = pd.concat([
    variants_exploded.drop(columns=["variants"]).reset_index(drop=True),
    variant_details,
], axis=1)


In [105]:
variants_cleaned_df.head()

,marketHashName,game,sold24h,priceUpdatedAt,market,price,quantity,capturedAt,variant,best,bestMarket,updatedAt,markets
0,Glock-18 | Gamma Doppler (Battle-Scarred),cs2,2.0,2026-11-15T00:08:04.575Z,avan,8018.0,51.0,2026-11-17T13:43:09.578Z,Emerald,18044,avan,2026-11-17T13:44:25.468Z,"[{'market': 'avan', 'price': 18044, 'quantity'..."
1,Glock-18 | Gamma Doppler (Battle-Scarred),cs2,2.0,2026-11-15T00:08:04.575Z,avan,8018.0,51.0,2026-11-17T13:43:09.578Z,Phase 1,6743,buff163,2026-11-17T13:44:25.468Z,"[{'market': 'buff163', 'price': 6743, 'quantit..."
2,Glock-18 | Gamma Doppler (Battle-Scarred),cs2,2.0,2026-11-15T00:08:04.575Z,avan,8018.0,51.0,2026-11-17T13:43:09.578Z,Phase 2,6600,buff163,2026-11-17T13:44:25.468Z,"[{'market': 'buff163', 'price': 6600, 'quantit..."
3,Glock-18 | Gamma Doppler (Battle-Scarred),cs2,2.0,2026-11-15T00:08:04.575Z,avan,8018.0,51.0,2026-11-17T13:43:09.578Z,Phase 3,6508,skinport,2026-11-17T13:44:25.468Z,"[{'market': 'skinport', 'price': 6508, 'quanti..."
4,Glock-18 | Gamma Doppler (Battle-Scarred),cs2,2.0,2026-11-15T00:08:04.575Z,avan,8018.0,51.0,2026-11-17T13:43:09.578Z,Phase 4,7023,marketcsgo,2026-11-17T13:44:25.468Z,"[{'market': 'marketcsgo', 'price': 7023, 'quan..."


# How to richtig alle Values kriegen

